## tl;dr
Seed 42 上，Candidate-DQN 相比 Equal-weight 降低总成本 1.29%，提高完成任务绿电覆盖 2.04 个百分点，完成率不变；但 Flexible 准时率下降 1.98 个百分点。单一 seed 只能用于诊断。

## Context & Methods
读取五份冻结策略评估 JSON，验证状态与配对轨迹元数据，然后生成宽表、长表、SLA 表及相对 Equal-weight 的差值。
### Key Assumptions
五份报告的 seed、任务轨迹、外生轨迹、拓扑、配置和依赖锁哈希必须一致。

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
ROOT = Path('.')
DATA = ROOT / 'artifacts/v1/evaluation/strategy_comparison_seed42'
wide = pd.read_csv(DATA / 'policy_metrics_wide.csv')
sla = pd.read_csv(DATA / 'sla_metrics.csv')
wide

## Data
宽表每行一个策略；SLA 表每行一个策略与 SLA 类型。

In [ ]:
wide[['policy_label', 'cost_yuan_per_completed_cpu_hour', 'completed_task_green_coverage', 'completion_rate', 'soft_preferred_on_time_rate', 'flexible_preferred_on_time_rate', 'active_wait_count']]

## Results
分别绘制成本、绿电、SLA 与等待任务，避免把不同单位放在同一坐标轴。

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
wide.plot.barh(x='policy_label', y='cost_yuan_per_completed_cpu_hour', ax=axes[0,0], legend=False, title='Cost per completed CPU-hour')
wide.plot.barh(x='policy_label', y='completed_task_green_coverage', ax=axes[0,1], legend=False, title='Completed-task green coverage')
wide.plot.barh(x='policy_label', y=['soft_preferred_on_time_rate','flexible_preferred_on_time_rate'], ax=axes[1,0], title='Preferred on-time rate')
wide.plot.barh(x='policy_label', y='active_wait_count', ax=axes[1,1], legend=False, title='Active-wait tasks')
plt.tight_layout()

## Takeaways
- DQN 在成本与绿电上同时优于 Equal-weight。
- Lowest-cost 与 Highest-green 是明显的单目标极端策略。
- DQN 的 Flexible 准时率退化需要在多 seed 上复核。
- 下一步应做至少 10 个未见 seed 的配对评估。